# sentiment-es — Fine-tuning BETO para sentimiento en español

Clasificacion de sentimiento (positivo / negativo / neutro).  
Modelo base: `dccuchile/bert-base-spanish-wwm-cased` (BETO) — BERT entrenado en corpus español nativo  
Dataset: `mteb/tweet_sentiment_multilingual` — tweets reales en español con anotacion humana

**Autor:** Lopez Insua — [github.com/lopezinsua](https://github.com/lopezinsua)

## 1. Setup & imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Carga del dataset + exploración

In [ ]:
raw = load_dataset("mteb/tweet_sentiment_multilingual", "spanish")
print(raw)
print("\nEjemplos:")
for ex in raw["train"].select(range(3)):
    print(ex)

In [ ]:
# Etiquetas vienen como strings ('0','1','2') — convertir a int
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

def cast_label(batch):
    batch["label"] = [int(l) for l in batch["label"]]
    return batch

raw = raw.map(cast_label, batched=True)

train_data = raw["train"]
test_data  = raw["test"]

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

from collections import Counter
counts = Counter(train_data["label"])
print("\nDistribucion de clases (train):")
for lid, cnt in sorted(counts.items()):
    print(f"  {ID2LABEL[lid]}: {cnt} ({cnt/len(train_data)*100:.1f}%)")

## 3. Tokenización

In [ ]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128, padding="max_length")

keep_cols = {"label"}
train_tok = train_data.map(tokenize, batched=True,
                            remove_columns=[c for c in train_data.column_names if c not in keep_cols])
test_tok  = test_data.map(tokenize,  batched=True,
                           remove_columns=[c for c in test_data.column_names  if c not in keep_cols])

train_tok.set_format("torch")
test_tok.set_format("torch")

print("Columnas disponibles:", train_tok.column_names)
print("Ejemplo tokenizado:", {k: v.shape for k, v in train_tok[0].items()})

## 4. Fine-tuning con Trainer

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("\nEntrenamiento completado.")
print(f"Loss final: {train_result.training_loss:.4f}")

## 5. Evaluación

In [ ]:
# Métricas globales
eval_results = trainer.evaluate()
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 macro: {eval_results['eval_f1']:.4f}")

In [ ]:
# Classification report completo
preds_output = trainer.predict(test_tok)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=[ID2LABEL[i] for i in sorted(ID2LABEL)]
))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=[ID2LABEL[i] for i in sorted(ID2LABEL)])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — roberta-sentiment-es")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Curva de loss train vs eval
history = trainer.state.log_history
train_losses = [(h["step"], h["loss"]) for h in history if "loss" in h and "eval_loss" not in h]
eval_losses  = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(8, 4))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="train loss", alpha=0.7)
if eval_losses:
    steps, elosses = zip(*eval_losses)
    ax.plot(steps, elosses, "o-", label="eval loss")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training vs Eval Loss")
ax.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

## 6. Inferencia

In [ ]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict(text: str) -> dict:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    label = model.config.id2label[probs.argmax().item()]
    return {
        "label": label,
        "confidence": round(probs.max().item(), 3),
        "scores": {model.config.id2label[i]: round(p.item(), 3) for i, p in enumerate(probs)}
    }

# Prueba rápida
print(predict("Me encanta este producto, es increíble"))

## 7. Publicación en HuggingFace Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Introduce tu token HF aquí

In [ ]:
HF_REPO = "lopezinsua/beto-sentiment-es"

trainer.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"Modelo publicado en: https://huggingface.co/{HF_REPO}")

## 8. Ejemplos finales

In [ ]:
ejemplos = [
    "Me encanta este producto, es increíble",
    "El servicio fue pésimo, nunca volvería",
    "El producto llegó en el plazo indicado",
    "Llevo esperando tres semanas y nada, vergonzoso",
    "Justo lo que necesitaba, muy recomendable",
    "No está mal, cumple con lo básico",
]

print(f"{'TEXTO':<50} {'LABEL':<12} {'CONF':>6}")
print("-" * 72)
for texto in ejemplos:
    res = predict(texto)
    print(f"{texto[:48]:<50} {res['label']:<12} {res['confidence']:>6.3f}")